<a href="https://colab.research.google.com/github/PeiHsiuLu/112-2-Programming-Language/blob/main/emooutput_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
model_path = (r"D:\dataset\model_5.h5")

In [ ]:
import cv2
import numpy as np
from tensorflow.keras.models import load_model

# Load the pre-trained model
new_model = load_model(model_path)

# Emotions dictionary
emotions = {
    0: "Angry", 1: "Fear", 2: "Happy",
    3: "Neutral", 4: "Sad", 5: "Surprise"
}

# Webcam capture setup
cap = cv2.VideoCapture(0)
if not cap.isOpened():
    raise IOError("Cannot open webcam")

# Function to handle drawing of emotion text
def handle_emotion(frame, status):
    text_size = cv2.getTextSize(status, cv2.FONT_HERSHEY_PLAIN, 1.5, thickness=1)[0]
    cv2.putText(frame, status, (10, 30), cv2.FONT_HERSHEY_PLAIN, 1.5, (0, 255, 0), 2)

# Main loop for frame capture and processing
while True:
    ret, frame = cap.read()
    if not ret:
        continue
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faceCascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
    faces = faceCascade.detectMultiScale(gray, 1.1, 4)
    emotion_detected = "None"  # Default if no face is detected

    for (x, y, w, h) in faces:
        face_roi = gray[y:y+h, x:x+w]
        face_roi = cv2.resize(face_roi, (224, 224))
        face_roi = cv2.cvtColor(face_roi, cv2.COLOR_GRAY2RGB)  # Convert grayscale to RGB
        face_roi = np.expand_dims(face_roi, axis=0) / 255.0

        try:
            predictions = new_model.predict(face_roi)
            emotion_index = np.argmax(predictions)
            if emotion_index in emotions:
                emotion_detected = emotions[emotion_index]
        except Exception as e:
            print(f"Error during model prediction: {e}")

    handle_emotion(frame, emotion_detected)  # Display the detected emotion on top left corner

    cv2.imshow('Face Emotion Recognition', frame)

    if cv2.waitKey(2) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

1/1 [==============================] - 0s 14ms/step


In [ ]:
import cv2
import numpy as np
from tensorflow.keras.models import load_model

# Load the pre-trained model
model_path = "D:\dataset\model.h5"  # 請將這裡替換為您模型的實際路徑
new_model = load_model(model_path)

# Emotions dictionary
emotions = {
    0: "Angry", 1: "Fear", 2: "Happy",
    3: "Neutral", 4: "Sad", 5: "Surprise"
}

# Load video
video_path = "D:\dataset\man.mp4"  # 替換為您的視頻文件路徑
cap = cv2.VideoCapture(video_path)
if not cap.isOpened():
    raise IOError("Cannot open video")

# Get video properties for output file
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
out_fps = cap.get(cv2.CAP_PROP_FPS)

# Define the codec and create VideoWriter object
out = cv2.VideoWriter("D:\dataset\manout1.mp4", cv2.VideoWriter_fourcc(*'mp4v'), out_fps, (frame_width, frame_height))

# Function to handle drawing of emotion text
def handle_emotion(frame, status):
    # Set the text position
    text_x = 100  # 水平位置
    text_y = 100  # 垂直位置

    # Set the text color
    text_color = (0, 255, 0)  # Green color

    # Drawing text background for better visibility
    cv2.rectangle(frame, (text_x, text_y - 20), (text_x + 180, text_y + 10), (0, 0, 0), -1)

    # Drawing the text
    cv2.putText(frame, status, (text_x, text_y), cv2.FONT_HERSHEY_COMPLEX, 1, text_color, 2)

# Main loop for frame capture and processing
while True:
    ret, frame = cap.read()
    if not ret:
        break
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faceCascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
    faces = faceCascade.detectMultiScale(gray, 1.1, 4)
    emotion_detected = "None"  # Default if no face is detected

    for (x, y, w, h) in faces:
        face_roi = gray[y:y+h, x:x+w]
        face_roi = cv2.resize(face_roi, (224, 224))
        face_roi = cv2.cvtColor(face_roi, cv2.COLOR_GRAY2RGB)  # Convert grayscale to RGB
        face_roi = np.expand_dims(face_roi, axis=0) / 255.0

        try:
            predictions = new_model.predict(face_roi)
            emotion_index = np.argmax(predictions)
            if emotion_index in emotions:
                emotion_detected = emotions[emotion_index]
        except Exception as e:
            print(f"Error during model prediction: {e}")

    handle_emotion(frame, emotion_detected)  # Draw the detected emotion on top left corner
    out.write(frame)  # Write the frame with annotations into the output video

# Release everything when job is finished
cap.release()
out.release()
cv2.destroyAllWindows()


1/1 [==============================] - 0s 16ms/step


In [ ]:
import subprocess

# 原始帶有音頻的視頻文件路徑
original_video_path = "D:\\dataset\\manout1.mp4"
# 不帶音頻的新視頻文件路徑
new_video_path = "D:\\dataset\\man.mp4"
# 輸出的最終視頻文件路徑
output_video_path = "D:\\dataset\\final_output.mp4"

# 命令行指令
command = f"ffmpeg -i {original_video_path} -i {new_video_path} -c:v copy -c:a aac -strict experimental {output_video_path}"

# 執行命令
subprocess.run(command, shell=True)

CompletedProcess(args='ffmpeg -i D:\\dataset\\manout1.mp4 -i D:\\dataset\\man.mp4 -c:v copy -c:a aac -strict experimental D:\\dataset\\final_output.mp4', returncode=0)